In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/dsn-bootcamp-qualification-hackathon-2026-ml-track/sample_submission.csv
/kaggle/input/competitions/dsn-bootcamp-qualification-hackathon-2026-ml-track/train.csv
/kaggle/input/competitions/dsn-bootcamp-qualification-hackathon-2026-ml-track/test.csv


In [2]:
import numpy as np
import pandas as pd
import optuna
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# 1. OUT-OF-FOLD TARGET ENCODER CLASS
# ==========================================
class OutOfFoldTargetEncoder:
    """
    Applies K-Fold Out-of-Fold Target Encoding with Bayesian M-estimate smoothing.
    Prevents target leakage and manages high-cardinality noise natively.
    """
    def __init__(self, categorical_cols, target_col, n_splits=5, m_smoothing=10, random_state=42):
        self.categorical_cols = categorical_cols
        self.target_col = target_col
        self.n_splits = n_splits
        self.m_smoothing = m_smoothing
        self.random_state = random_state
        self.global_means = {}
        self.category_mappings = {}

    def fit_transform(self, train_df):
        df = train_df.copy()
        kf = KFold(n_splits=self.n_splits, shuffle=True, random_state=self.random_state)
        
        for col in self.categorical_cols:
            df[f"{col}_te"] = np.nan
            self.global_means[col] = df[self.target_col].mean()
            
            # K-Fold Out-of-Fold Encoding for Train Data
            for train_idx, val_idx in kf.split(df):
                X_tr, X_va = df.iloc[train_idx], df.iloc[val_idx]
                
                stats = X_tr.groupby(col)[self.target_col].agg(['count', 'mean'])
                # Bayesian M-estimate smoothing formula
                smooth = (stats['count'] * stats['mean'] + self.m_smoothing * self.global_means[col]) / (stats['count'] + self.m_smoothing)
                df.loc[val_idx, f"{col}_te"] = X_va[col].map(smooth)
            
            # Fallback for unseen categories
            df[f"{col}_te"] = df[f"{col}_te"].fillna(self.global_means[col])
            
            # Store full-dataset encodings for test data mapping
            full_stats = df.groupby(col)[self.target_col].agg(['count', 'mean'])
            self.category_mappings[col] = (full_stats['count'] * full_stats['mean'] + self.m_smoothing * self.global_means[col]) / (full_stats['count'] + self.m_smoothing)
            
        return df

    def transform(self, test_df):
        df = test_df.copy()
        for col in self.categorical_cols:
            if col in self.category_mappings:
                df[f"{col}_te"] = df[col].map(self.category_mappings[col])
                df[f"{col}_te"] = df[f"{col}_te"].fillna(self.global_means[col])
        return df

# ==========================================
# 2. CUSTOM HUBER LOSS FUNCTION
# ==========================================
def get_huber_objective(delta):
    """
    Returns a custom Huber objective for LightGBM.
    Treats errors < delta as L2 (MSE) and > delta as L1 (MAE) to ignore extreme outliers.
    """
    def custom_huber(y_true, y_pred):
        residual = y_true - y_pred
        grad = np.where(np.abs(residual) <= delta, -residual, -delta * np.sign(residual))
        # Strictly positive hessians (1e-4) required for LightGBM Newton splits
        hess = np.where(np.abs(residual) <= delta, 1.0, 1e-4)
        return grad, hess
    return custom_huber

# ==========================================
# 3. LOAD & CLEAN DATA
# ==========================================
train_path = '/kaggle/input/competitions/dsn-bootcamp-qualification-hackathon-2026-ml-track/train.csv'
test_path = '/kaggle/input/competitions/dsn-bootcamp-qualification-hackathon-2026-ml-track/test.csv'

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
test_ids = test['id']

train['is_train'] = 1
test['is_train'] = 0
test['total_sales'] = np.nan
df = pd.concat([train, test], ignore_index=True)

df['product_category'] = df['product_category'].astype(str).str.lower().str.strip()
fat_mapping = {'lf': 'low_fat', 'low fat': 'low_fat', 'low_fat': 'low_fat', 'regular': 'regular', 'reg': 'regular'}
df['fat_content'] = df['fat_content'].astype(str).str.lower().str.strip().map(fat_mapping).fillna('regular')

df['shelf_visibility'] = df['shelf_visibility'].replace(0, np.nan)
df['shelf_visibility'] = df.groupby('product_code')['shelf_visibility'].transform(lambda x: x.fillna(x.median()))
df['shelf_visibility'] = df['shelf_visibility'].fillna(df['shelf_visibility'].median())

df['product_weight_kg'] = df.groupby('product_code')['product_weight_kg'].transform(lambda x: x.fillna(x.median()))
df['product_weight_kg'] = df['product_weight_kg'].fillna(df['product_weight_kg'].median())

# ==========================================
# 4. BASE FEATURE ENGINEERING
# ==========================================
df['category_mean_price'] = df.groupby('product_category')['product_price'].transform('mean')
df['price_ratio_to_category'] = df['product_price'] / (df['category_mean_price'] + 1e-5)
df['price_per_kg'] = df['product_price'] / df['product_weight_kg']
df['visibility_x_price'] = df['shelf_visibility'] * df['product_price']

df['age_tier'] = pd.qcut(df['store_age_years'], q=3, labels=['New', 'Established', 'Old'], duplicates='drop')
df['weight_tier'] = pd.qcut(df['product_weight_kg'], q=3, labels=['Light', 'Medium', 'Heavy'])
df['price_tier'] = pd.qcut(df['product_price'], q=4, labels=['Cheap', 'Fair', 'Expensive', 'Premium'])

simple_text_cols = ['fat_content', 'product_category', 'store_location_tier', 'store_format', 'store_size', 'age_tier', 'weight_tier', 'price_tier']
df = pd.get_dummies(df, columns=simple_text_cols, drop_first=True, dtype=int)

train_clean = df[df['is_train'] == 1].drop(['is_train', 'id'], axis=1)
test_clean = df[df['is_train'] == 0].drop(['is_train', 'total_sales', 'id'], axis=1)

# Apply Out-of-Fold Target Encoding (Replaces LabelEncoder)
high_cardinality_cols = ['store_code', 'product_code']
encoder = OutOfFoldTargetEncoder(categorical_cols=high_cardinality_cols, target_col='total_sales', m_smoothing=15)
train_clean = encoder.fit_transform(train_clean)
test_clean = encoder.transform(test_clean)

# Drop original raw categorical strings now that we have dense Target Encoded versions
train_clean = train_clean.drop(columns=high_cardinality_cols)
test_clean = test_clean.drop(columns=high_cardinality_cols)

X = train_clean.drop('total_sales', axis=1)
y = train_clean['total_sales']
X_test = test_clean

# Calculate dynamic Delta for Huber Loss (1.5 Standard Deviations)
dynamic_delta = y.std() * 1.5
huber_obj = get_huber_objective(delta=dynamic_delta)

# ==========================================
# 5. THE MICRO-SQUEEZE (OPTUNA OPTIMIZATION)
# ==========================================
print("Starting Deep Optuna Micro-Squeeze with Custom Huber Loss...")
def objective(trial):
    param = {
        'metric': 'rmse',
        'verbosity': -1,
        'random_state': 42,
        'n_estimators': 3000, 
        'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.015), # Extremely slow burn
        'num_leaves': trial.suggest_int('num_leaves', 15, 80),
        'max_depth': trial.suggest_int('max_depth', 4, 10),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 80),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True)
    }
    
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    fold_rmses = []
    
    for train_idx, val_idx in kf.split(X, y):
        X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
        X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]
        
        model = lgb.LGBMRegressor(**param)
        model.set_params(objective=huber_obj) # Inject Custom Objective
        
        model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], eval_metric='rmse', callbacks=[lgb.early_stopping(150, verbose=False)])
        
        preds = model.predict(X_va)
        fold_rmses.append(np.sqrt(mean_squared_error(y_va, preds)))
        
    return np.mean(fold_rmses)

# Run robust search
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=75) # 75 deep trials

# ==========================================
# 6. SEED BLENDING & FINAL PREDICTIONS
# ==========================================
print("\n--- Training Final Champion with Seed Blending ---")
best_params = study.best_params
best_params.update({'metric': 'rmse', 'verbosity': -1, 'n_estimators': 5000})

seed_list = [42, 2026, 999]
final_test_preds = np.zeros(len(X_test))
oof_preds = np.zeros(len(X))

for seed in seed_list:
    print(f"Executing training path for random seed: {seed}...")
    best_params['random_state'] = seed
    kf = KFold(n_splits=5, shuffle=True, random_state=seed)
    
    seed_preds = np.zeros(len(X_test))
    for train_idx, val_idx in kf.split(X, y):
        X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
        X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]
        
        model = lgb.LGBMRegressor(**best_params)
        model.set_params(objective=huber_obj)
        
        model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], eval_metric='rmse', callbacks=[lgb.early_stopping(200, verbose=False)])
        
        oof_preds[val_idx] += model.predict(X_va) / len(seed_list)
        seed_preds += model.predict(X_test) / kf.n_splits
        
    final_test_preds += seed_preds / len(seed_list)

# Post-processing boundary clip
final_test_preds = np.clip(final_test_preds, a_min=0, a_max=None)

final_cv_rmse = np.sqrt(mean_squared_error(y, oof_preds))
print(f"\nFinal Seed-Blended OOF RMSE: {final_cv_rmse:.4f}")

submission = pd.DataFrame({'id': test_ids, 'total_sales': final_test_preds})
submission.to_csv('submission_huber_oof.csv', index=False)
print("Research Blueprint submission generated successfully!")

[I 2026-09-27 01:58:17,585] A new study created in memory with name: no-name-1b504a9b-6ba3-49dd-b683-bf0d7f5165f8


Starting Deep Optuna Micro-Squeeze with Custom Huber Loss...


[I 2026-09-27 01:58:26,482] Trial 0 finished with value: 1086.8284700315678 and parameters: {'learning_rate': 0.003908691064567139, 'num_leaves': 19, 'max_depth': 6, 'min_child_samples': 36, 'subsample': 0.9258497283204357, 'colsample_bytree': 0.9945332967754167, 'reg_alpha': 1.6913944005321948e-08, 'reg_lambda': 0.056883064749749454}. Best is trial 0 with value: 1086.8284700315678.
[I 2026-09-27 01:58:35,582] Trial 1 finished with value: 1101.9706297765533 and parameters: {'learning_rate': 0.007123233670243053, 'num_leaves': 50, 'max_depth': 9, 'min_child_samples': 27, 'subsample': 0.7161304341281969, 'colsample_bytree': 0.8912351591237704, 'reg_alpha': 9.166248822895399e-07, 'reg_lambda': 0.3201587818868172}. Best is trial 0 with value: 1086.8284700315678.
[I 2026-09-27 01:58:58,984] Trial 2 finished with value: 1082.0021699608028 and parameters: {'learning_rate': 0.0011952481267191064, 'num_leaves': 45, 'max_depth': 5, 'min_child_samples': 11, 'subsample': 0.7727742815393366, 'colsa


--- Training Final Champion with Seed Blending ---
Executing training path for random seed: 42...
Executing training path for random seed: 2026...
Executing training path for random seed: 999...

Final Seed-Blended OOF RMSE: 1074.0071
Research Blueprint submission generated successfully!
